In [1]:
# Step 1: Imports

import os
import glob
import zipfile
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import joblib
import time

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [2]:
# Step 2: Locate and load the dataset file

candidates = glob.glob("/kaggle/input/**/master_dataset.csv", recursive=True)
candidates += glob.glob("/kaggle/input/**/master_dataset.zip", recursive=True)

if not candidates:
    raise FileNotFoundError(
        "Could not find master_dataset.csv or master_dataset.zip under /kaggle/input/. "
        "Make sure you've added the dataset to this notebook (Add Data -> your dataset)."
    )

DATA_PATH = candidates[0]
print("Using data file:", DATA_PATH)

Using data file: /kaggle/input/datasets/pes2ug23cs197/capstone-dataset/master_dataset.csv


In [3]:
# Step 3: Memory-efficient load

# The symptom columns are all binary (0/1), so we load them as int8 instead of
# the pandas default int64 — this cuts memory usage by roughly 8x.

import re

header_df = pd.read_csv(DATA_PATH, nrows=0)
all_columns = header_df.columns.tolist()
target_col = "disease"
original_feature_cols = [c for c in all_columns if c != target_col]

dtype_map = {col: "int8" for col in original_feature_cols}
dtype_map[target_col] = "category"

start = time.time()
df = pd.read_csv(DATA_PATH, dtype=dtype_map)
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]:,} columns in {time.time()-start:.1f}s")
print("Memory usage: {:.2f} MB".format(df.memory_usage(deep=True).sum() / 1e6))

# LightGBM stores its model as JSON internally, so feature names can't contain
# JSON special characters (parentheses, quotes, colons, etc. We sanitize the names
# once here, and keep a mapping back to the original text for anything that
# needs the human-readable version later (e.g. displaying symptom names in
# the app's UI).

def sanitize_column_name(name):
    clean = re.sub(r"[^0-9a-zA-Z_]", "_", name)
    clean = re.sub(r"_+", "_", clean).strip("_")
    return clean if clean else "feature"

sanitized_feature_cols = []
seen_names = {}
for name in original_feature_cols:
    clean = sanitize_column_name(name)
    if clean in seen_names:
        seen_names[clean] += 1
        clean = f"{clean}_{seen_names[clean]}"
    else:
        seen_names[clean] = 0
    sanitized_feature_cols.append(clean)

rename_map = dict(zip(original_feature_cols, sanitized_feature_cols))
df = df.rename(columns=rename_map)

feature_cols = sanitized_feature_cols
sanitized_to_original = dict(zip(sanitized_feature_cols, original_feature_cols))

print(f"Sanitized {len(feature_cols)} feature column names for LightGBM compatibility.")

Loaded 190,622 rows x 1,757 columns in 21.7s
Memory usage: 335.22 MB
Sanitized 1756 feature column names for LightGBM compatibility.


In [4]:
# Step 4: Filter out rare disease classes

# Many classes have very few samples (some just 1), which breaks stratified
# splitting and gives the model nothing to learn from. We drop classes below
# a minimum count in the FULL dataset before sampling.

MIN_SAMPLES_PER_CLASS = 100  

class_counts = df[target_col].value_counts()
valid_classes = class_counts[class_counts >= MIN_SAMPLES_PER_CLASS].index
df_filtered = df[df[target_col].isin(valid_classes)].copy()
df_filtered[target_col] = df_filtered[target_col].cat.remove_unused_categories()

print(f"Classes before filtering: {df[target_col].nunique()}")
print(f"Classes after filtering (>= {MIN_SAMPLES_PER_CLASS} samples): {df_filtered[target_col].nunique()}")
print(f"Rows before filtering: {len(df):,} | Rows after filtering: {len(df_filtered):,}")

del df  # free memory

Classes before filtering: 1117
Classes after filtering (>= 100 samples): 348
Rows before filtering: 190,622 | Rows after filtering: 179,513


In [5]:
# Step 5: Sample a FIXED number of rows PER CLASS (not a fixed total)

# Sampling a fixed total and splitting it proportionally across ~500-900
# classes starves the smaller classes down to just a handful of rows, which
# is what causes severe train/test overfitting — the model has too few
# examples per class to generalize from. Instead, we cap every class at the
# SAME number of rows, so every class gets a reasonable, comparable amount
# of train and test data.

N_PER_CLASS = 200  

def capped_sample_per_class(data, target_col, n_per_class, seed=42):
    # NOTE: iterating groups directly (instead of groupby().apply()) avoids the
    # pandas FutureWarning about grouping columns being included/excluded, and
    # is a bit faster too.
    parts = []
    for _, g in data.groupby(target_col, observed=True):
        n = min(n_per_class, len(g))
        parts.append(g.sample(n=n, random_state=seed))
    return pd.concat(parts, ignore_index=False)

df_sample = capped_sample_per_class(df_filtered, target_col, N_PER_CLASS, RANDOM_STATE)
df_sample = df_sample.sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)  # shuffle

print(f"Final sample size: {len(df_sample):,} rows across {df_sample[target_col].nunique()} classes")
print("Rows per class - min:", df_sample[target_col].value_counts().min(),
      "| max:", df_sample[target_col].value_counts().max())

del df_filtered

Final sample size: 65,917 rows across 348 classes
Rows per class - min: 100 | max: 200


In [6]:
# Step 6: Encode labels and split train/test

# X is kept as a pandas DataFrame (not .values) so LightGBM's internal feature
# names stay consistent between fit() and predict() — this avoids the
# "X does not have valid feature names" warning, and gives readable feature
# names in the classification report / feature importances.

X = df_sample[feature_cols]  # DataFrame, not .values
y_raw = df_sample[target_col].astype(str).values

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f"Train size: {X_train.shape[0]:,} | Test size: {X_test.shape[0]:,}")
print(f"Number of classes: {len(label_encoder.classes_)}")

Train size: 52,733 | Test size: 13,184
Number of classes: 348


In [7]:
# Step 7: Baseline — Logistic Regression

# Fast sanity check on how separable the classes are before investing time
# in LightGBM tuning.

start = time.time()
baseline = LogisticRegression(
    max_iter=1000, n_jobs=-1, random_state=RANDOM_STATE
)
baseline.fit(X_train, y_train)
train_time = time.time() - start

train_acc = accuracy_score(y_train, baseline.predict(X_train))
test_acc = accuracy_score(y_test, baseline.predict(X_test))

print(f"[Logistic Regression] trained in {train_time:.1f}s")
print(f"Train accuracy: {train_acc:.4f} | Test accuracy: {test_acc:.4f}")
print(f"Train-test gap: {abs(train_acc - test_acc):.4f}")

[Logistic Regression] trained in 167.3s
Train accuracy: 0.8846 | Test accuracy: 0.8623
Train-test gap: 0.0223


In [8]:
# Step 8: Main model — LightGBM (multiclass)

# Regularized settings (moderate depth/leaves, L1/L2 penalties, min samples
# per leaf) to keep the train-test accuracy gap small, plus early stopping
# so we don't overfit by training too many rounds.

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.15, stratify=y_train, random_state=RANDOM_STATE
)

# Diagnostic: confirm every class has enough train examples after the split.
train_class_counts = pd.Series(y_tr).value_counts()
print("Train samples per class — min:", train_class_counts.min(),
      "| median:", int(train_class_counts.median()),
      "| max:", train_class_counts.max())

lgbm = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=len(label_encoder.classes_),
    n_estimators=900,
    learning_rate=0.02,        
    num_leaves=10,              
    max_depth=5,
    min_child_samples=40,       
    reg_alpha=0.5,
    reg_lambda=0.5,
    subsample=0.6,
    subsample_freq=1,
    colsample_bytree=0.6,
    extra_trees=True,           
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=-1
)

start = time.time()
lgbm.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    eval_metric="multi_logloss",
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(period=50)],
)
train_time = time.time() - start

train_acc = accuracy_score(y_train, lgbm.predict(X_train))
test_acc = accuracy_score(y_test, lgbm.predict(X_test))

print(f"\n[LightGBM] trained in {train_time:.1f}s | best iteration: {lgbm.best_iteration_}")
print(f"Train accuracy: {train_acc*100:.2f}% | Test accuracy: {test_acc*100:.2f}%")
print(f"Train-test gap: {abs(train_acc - test_acc)*100:.2f}%")

Train samples per class — min: 68 | median: 136 | max: 136
Training until validation scores don't improve for 50 rounds
[50]	valid_0's multi_logloss: 1.55596
[100]	valid_0's multi_logloss: 0.958909
[150]	valid_0's multi_logloss: 0.723585
[200]	valid_0's multi_logloss: 0.611221
[250]	valid_0's multi_logloss: 0.549471
[300]	valid_0's multi_logloss: 0.511805
[350]	valid_0's multi_logloss: 0.487093
[400]	valid_0's multi_logloss: 0.469881
[450]	valid_0's multi_logloss: 0.457359
[500]	valid_0's multi_logloss: 0.447869
[550]	valid_0's multi_logloss: 0.440486
[600]	valid_0's multi_logloss: 0.435169
[650]	valid_0's multi_logloss: 0.430819
[700]	valid_0's multi_logloss: 0.427264
[750]	valid_0's multi_logloss: 0.424815
[800]	valid_0's multi_logloss: 0.422203
[850]	valid_0's multi_logloss: 0.420399
[900]	valid_0's multi_logloss: 0.419201
Did not meet early stopping. Best iteration is:
[900]	valid_0's multi_logloss: 0.419201

[LightGBM] trained in 1404.0s | best iteration: 900
Train accuracy: 91.10

In [9]:
# Step 9: Top-3 disease prediction function

def predict_top3(symptom_vector, model=lgbm, encoder=label_encoder, columns=feature_cols):
    """
    symptom_vector: 1D array-like of 0/1 values, length = len(feature_cols),
                     in the SAME COLUMN ORDER as feature_cols.
    Returns: list of (disease_name, probability) tuples, top 3, sorted desc.
    """
    # Wrapped in a DataFrame with the correct column names so it matches what
    # the model was fitted on — keeps predict() warning-free and correct.
    X_input = pd.DataFrame([symptom_vector], columns=columns)
    proba = model.predict_proba(X_input)[0]
    top3_idx = np.argsort(proba)[-3:][::-1]
    return [(encoder.classes_[i], float(proba[i])) for i in top3_idx]

# Quick demo using a real test-set row
demo_row = X_test.iloc[0].values
demo_result = predict_top3(demo_row)
print("Demo top-3 prediction for a test sample:")
for disease, prob in demo_result:
    print(f"  {disease}: {prob:.3f}")
print("Actual disease:", label_encoder.classes_[y_test[0]])

Demo top-3 prediction for a test sample:
  Dysthymic Disorder: 0.596
  Depression: 0.365
  Impulse Control Disorder: 0.010
Actual disease: Dysthymic Disorder


In [10]:
# Step 10: Save artifacts for the app (model, label encoder, feature list)

os.makedirs("/kaggle/working/artifacts", exist_ok=True)

joblib.dump(lgbm, "/kaggle/working/artifacts/lgbm_model.pkl")
joblib.dump(label_encoder, "/kaggle/working/artifacts/label_encoder.pkl")
joblib.dump(feature_cols, "/kaggle/working/artifacts/feature_columns.pkl")
joblib.dump(sanitized_to_original, "/kaggle/working/artifacts/feature_name_mapping.pkl")

print("Saved LightGBM Model")

Saved LightGBM Model
